# Jina Reranker — 긴 문서 처리에 강한 재순위화

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **Jina Reranker**의 차별화 특징(최대 8K 토큰 지원)을 이해하기
- `JinaRerank` 클래스로 재순위화 파이프라인 구현하기
- Cross-Encoder, Cohere, Jina 세 가지 Reranker를 비교해 적절한 선택 기준 파악하기

## 사전 지식

- 01-Cross-Encoder-Reranker, 02-Cohere-Reranker의 Two-Stage Retrieval 개념
- Jina API 키 발급 방법 (`JINA_API_KEY` 환경변수 필요)

---

## Reranker 3종 비교

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> VR[벡터 검색<br/>OpenAI + FAISS<br/>k=10]:::process
    VR --> JR[Jina Rerank API<br/>최대 8K 토큰<br/>top_n=3]:::external
    JR --> R[관련성 점수 포함<br/>상위 3개 문서]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

| 특징 | Jina Reranker | Cohere Rerank | Cross-Encoder |
|------|:---:|:---:|:---:|
| 최대 토큰 | **8K** | 4K | 512 |
| 속도 | 빠름 | 빠름 | 보통 |
| 비용 | $ | $$ | 무료 |
| 추천 용도 | **긴 문서** | 프로덕션 | 개발·테스트 |

> **실무 팁**: 논문, 보고서, 기술 문서처럼 청크 크기가 2000자 이상인 경우 Jina Reranker가 잘려나가는 현상 없이 전체를 분석할 수 있어요.

> 🔑 **핵심 개념**: Cross-Encoder와 Cohere는 입력 토큰이 512~4K로 제한되어 있어 긴 문서를 처리하면 내용이 잘립니다. Jina는 8K 토큰을 지원하므로, 약 3000~4000자 이상의 긴 청크를 사용하는 RAG 시스템에 적합합니다.

> 💡 **실무 팁**: 3개 Reranker 모두 `relevance_score`를 `metadata`에 추가합니다. 이 점수를 활용해 "관련성이 특정 임계값 이하인 문서는 LLM에 전달하지 않는" 추가 필터링도 가능해요.

## 1. 환경 설정

### 1.1 Jina API 키 발급

1. [Jina AI 웹사이트](https://jina.ai/reranker)에서 API 키 발급
2. `.env` 파일에 API 키 추가:

```bash
JINA_API_KEY="your-jina-api-key"
```

### 1.2 Jina Reranker 모델

- `jina-reranker-v2-base-multilingual`: 다국어 기본 모델 (권장)
- `jina-reranker-v1-base-en`: 영어 특화 모델
- `jina-reranker-v1-turbo-en`: 빠른 속도 (영어)


In [11]:
# 필요한 라이브러리 import
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import JinaRerank

# 환경 변수 로드
load_dotenv()
JINA_API_KEY = os.getenv("JINA_API_KEY")
if not JINA_API_KEY:
    print("⚠️ JINA_API_KEY가 없어 Jina Reranker 예제를 완전히 실행할 수 없습니다.")

## 2. 문서 출력 도우미 함수


In [12]:
def pretty_print_docs(docs, show_scores=True):
    """검색된 문서를 보기 좋게 출력하는 함수"""
    print(f"\n{'=' * 100}")
    print(f"총 {len(docs)}개 문서 검색됨")
    print(f"{'=' * 100}\n")
    
    for i, doc in enumerate(docs, 1):
        print(f"[문서 {i}]")
        
        # Jina Reranker도 relevance_score를 metadata에 추가
        if show_scores and 'relevance_score' in doc.metadata:
            score = doc.metadata['relevance_score']
            print(f"관련성 점수: {score:.4f}")
        
        print(f"내용: {doc.page_content}")
        print(f"{'─' * 100}\n")


## 3. 기본 검색 시스템 구축


In [13]:
# 1단계: 문서 로드
documents = TextLoader("./data/appendix-keywords.txt").load()

print(f"✅ 문서 로드 완료")
print(f"   - 문서 수: {len(documents)}")
print(f"   - 총 길이: {len(documents[0].page_content)} 문자")


✅ 문서 로드 완료
   - 문서 수: 1
   - 총 길이: 5733 문자


In [14]:
# 2단계: 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

texts = text_splitter.split_documents(documents)

print(f"✅ 문서 분할 완료")
print(f"   - 청크 수: {len(texts)}")


✅ 문서 분할 완료
   - 청크 수: 15


In [15]:
# 3단계: 벡터스토어 생성 (OpenAI 임베딩 사용)
vectorstore = FAISS.from_documents(texts, OpenAIEmbeddings())
base_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}  # 초기 검색: 10개 문서
)

print("✅ 벡터스토어 생성 완료")
print(f"   - 임베딩: OpenAI")
print(f"   - 초기 검색 수: 10개")


✅ 벡터스토어 생성 완료
   - 임베딩: OpenAI
   - 초기 검색 수: 10개


In [16]:
# ---------------------------------------------------
# Reranker 적용 전 기본 벡터 검색 결과 확인
# ---------------------------------------------------
# ============================================================
# TODO: base_retriever로 쿼리를 검색하고 결과를 출력하세요
# 힌트: invoke(query) 호출 후 pretty_print_docs(docs, show_scores=False)
# 예상 결과: 10개 문서가 OpenAI 임베딩 유사도 순으로 출력됨
# ============================================================

# 기본 검색 테스트
query = "구조화되지 않은 데이터를 정리하여 분석하는 방법은?"

print(f"\n🔍 검색 쿼리: {query}\n")

# Reranker 적용 전 기본 검색
docs_before = base_retriever.invoke(query)

print("📊 [Reranker 적용 전] 벡터 유사도 기반 검색 결과:")
pretty_print_docs(docs_before, show_scores=False)


🔍 검색 쿼리: 구조화되지 않은 데이터를 정리하여 분석하는 방법은?

📊 [Reranker 적용 전] 벡터 유사도 기반 검색 결과:

총 10개 문서 검색됨

[문서 1]
내용: Parser

정의: 파서는 주어진 데이터(문자열, 파일 등)를 분석하여 구조화된 형태로 변환하는 도구입니다. 이는 프로그래밍 언어의 구문 분석이나 파일 데이터 처리에 사용됩니다.
예시: HTML 문서를 구문 분석하여 웹 페이지의 DOM 구조를 생성하는 것은 파싱의 한 예입니다.
연관키워드: 구문 분석, 컴파일러, 데이터 처리

TF-IDF (Term Frequency-Inverse Document Frequency)

정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.
예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.
연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝

Deep Learning
────────────────────────────────────────────────────────────────────────────────────────────────────

[문서 2]
내용: Deep Learning

정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다.
예시: 이미지 인식, 음성 인식, 자연어 처리 등에서 딥러닝 모델이 활용됩니다.
연관키워드: 인공신경망, 머신러닝, 데이터 분석

Schema

정의: 스키마는 데이터베이스나 파일의 구조를 정의하는 것으로, 데이터가 어떻게 저장되고 조직되는지에 대한 청사진을 제공합니다.
예시: 관계형 데이터베이스의 테이블 스키마는 열 이름, 데이터 타입, 키 제약 조건 등을 정의합니다.
연관키워드: 데이터베이스, 데이터 모델링, 데이터 관리

DataFrame
─────────────

## 4. Jina Reranker 적용

Jina AI의 Rerank API를 사용하여 검색 결과를 재정렬합니다.


In [17]:
# ---------------------------------------------------
# Jina Reranker 설정 — 긴 문서 처리 능력 활용
# ---------------------------------------------------
# ============================================================
# TODO: JinaRerank와 ContextualCompressionRetriever로 재정렬 파이프라인을 만드세요
# 힌트: JinaRerank(model="jina-reranker-v2-base-multilingual", top_n=3)
# 예상 결과: "Jina Reranker 설정 완료" 출력 (최대 8K 토큰 지원 표시)
# ============================================================

if not JINA_API_KEY:
    compression_retriever = None
    print("⚠️ JINA_API_KEY가 없어 Jina Reranker 단계를 건너뜁니다.")
else:
    # 1단계: Jina Reranker 초기화
    # jina-reranker-v2-base-multilingual: 최대 8K 토큰 지원 다국어 Reranker
    compressor = JinaRerank(
        model="jina-reranker-v2-base-multilingual",
        top_n=3,  # 10개 후보에서 상위 3개만 최종 반환
        jina_api_key=JINA_API_KEY,
    )

    # 2단계: 압축 검색기 생성
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=base_retriever
    )

    print("✅ Jina Reranker 설정 완료")
    print(f"   - 모델: jina-reranker-v2-base-multilingual")
    print(f"   - 최대 토큰 길이: 8K")
    print(f"   - 초기 검색: 10개 → 최종 반환: 3개")

✅ Jina Reranker 설정 완료
   - 모델: jina-reranker-v2-base-multilingual
   - 최대 토큰 길이: 8K
   - 초기 검색: 10개 → 최종 반환: 3개


In [18]:
# ---------------------------------------------------
# Reranker 적용 후 검색 결과 확인
# ---------------------------------------------------
# ============================================================
# TODO: compression_retriever로 같은 쿼리를 검색하고 관련성 점수를 확인하세요
# 힌트: invoke(query) 후 pretty_print_docs(docs, show_scores=True)
# 예상 결과: 3개 문서와 Jina 관련성 점수 출력
# ============================================================

if compression_retriever is None:
    docs_after = []
    print("⚠️ Jina Reranker가 설정되지 않아 재정렬 예제를 건너뜁니다.")
else:
    # Reranker 적용 검색
    print(f"🔍 검색 쿼리: {query}\n")

    docs_after = compression_retriever.invoke(query)

    print("🎯 [Reranker 적용 후] Jina Rerank 결과:")
    pretty_print_docs(docs_after, show_scores=True)

🔍 검색 쿼리: 구조화되지 않은 데이터를 정리하여 분석하는 방법은?

🎯 [Reranker 적용 후] Jina Rerank 결과:

총 3개 문서 검색됨

[문서 1]
관련성 점수: 0.1907
내용: Parser

정의: 파서는 주어진 데이터(문자열, 파일 등)를 분석하여 구조화된 형태로 변환하는 도구입니다. 이는 프로그래밍 언어의 구문 분석이나 파일 데이터 처리에 사용됩니다.
예시: HTML 문서를 구문 분석하여 웹 페이지의 DOM 구조를 생성하는 것은 파싱의 한 예입니다.
연관키워드: 구문 분석, 컴파일러, 데이터 처리

TF-IDF (Term Frequency-Inverse Document Frequency)

정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.
예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.
연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝

Deep Learning
────────────────────────────────────────────────────────────────────────────────────────────────────

[문서 2]
관련성 점수: 0.1755
내용: DataFrame

정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 분석 및 처리에 사용됩니다.
예시: 판다스 라이브러리에서 DataFrame은 다양한 데이터 타입의 열을 가질 수 있으며, 데이터 조작과 분석을 용이하게 합니다.
연관키워드: 데이터 분석, 판다스, 데이터 처리

Attention 메커니즘

정의: Attention 메커니즘은 딥러닝에서 중요한 정보에 더 많은 '주의'를 기울이도록 하는 기법입니다. 이는 주로 시퀀스 데이터(예: 텍스트, 시계열 데이터)에서 사용됩니다.
예시: 번역 모델에서 Attention 메커니즘은 입력 문장의 중요한 

## 5. 결과 비교 및 분석


In [19]:
# ---------------------------------------------------
# Jina Reranker 적용 전후 결과 비교 분석
# ---------------------------------------------------
if compression_retriever is None:
    print("⚠️ Jina Reranker가 없어 효과 분석을 건너뜁니다.")
else:
    print("\n" + "=" * 100)
    print("📊 Jina Reranker 효과 분석")
    print("=" * 100)

    print(f"\n[검색 쿼리]")
    print(f"  {query}")

    print(f"\n[Reranker 적용 전] - 상위 3개")
    for i, doc in enumerate(docs_before[:3], 1):
        preview = doc.page_content.replace('\n', ' ')[:100]
        print(f"  {i}. {preview}...")

    print(f"\n[Reranker 적용 후] - 상위 3개 + 관련성 점수")
    for i, doc in enumerate(docs_after, 1):
        preview = doc.page_content.replace('\n', ' ')[:100]
        score = doc.metadata.get('relevance_score', 0)
        print(f"  {i}. [점수: {score:.4f}] {preview}...")

    print("\n💡 Jina Reranker의 강점:")
    print("  ✅ 긴 문서 처리: 최대 8K 토큰 (다른 모델보다 2배)")
    print("  ✅ 빠른 속도: 최적화된 추론 엔진")
    print("  ✅ 다국어 지원: 한국어 포함 다양한 언어")
    print("  ✅ 합리적 가격: 경쟁력 있는 API 요금")


📊 Jina Reranker 효과 분석

[검색 쿼리]
  구조화되지 않은 데이터를 정리하여 분석하는 방법은?

[Reranker 적용 전] - 상위 3개
  1. Parser  정의: 파서는 주어진 데이터(문자열, 파일 등)를 분석하여 구조화된 형태로 변환하는 도구입니다. 이는 프로그래밍 언어의 구문 분석이나 파일 데이터 처리에 사용됩니다....
  2. Deep Learning  정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다. 예시...
  3. Open Source  정의: 오픈 소스는 소스 코드가 공개되어 누구나 자유롭게 사용, 수정, 배포할 수 있는 소프트웨어를 의미합니다. 이는 협업과 혁신을 촉진하는 데 중요한 역할...

[Reranker 적용 후] - 상위 3개 + 관련성 점수
  1. [점수: 0.1907] Parser  정의: 파서는 주어진 데이터(문자열, 파일 등)를 분석하여 구조화된 형태로 변환하는 도구입니다. 이는 프로그래밍 언어의 구문 분석이나 파일 데이터 처리에 사용됩니다....
  2. [점수: 0.1755] DataFrame  정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 분석 및 처리에 사용됩니다. 예시: 판다스 라이브러리에서 DataFra...
  3. [점수: 0.1571] Open Source  정의: 오픈 소스는 소스 코드가 공개되어 누구나 자유롭게 사용, 수정, 배포할 수 있는 소프트웨어를 의미합니다. 이는 협업과 혁신을 촉진하는 데 중요한 역할...

💡 Jina Reranker의 강점:
  ✅ 긴 문서 처리: 최대 8K 토큰 (다른 모델보다 2배)
  ✅ 빠른 속도: 최적화된 추론 엔진
  ✅ 다국어 지원: 한국어 포함 다양한 언어
  ✅ 합리적 가격: 경쟁력 있는 API 요금


## 6. 다양한 쿼리 테스트


In [20]:
# ---------------------------------------------------
# 다양한 쿼리로 Jina Reranker 성능 검증
# ---------------------------------------------------
# ============================================================
# TODO: 여러 쿼리를 순회하며 Jina Reranker로 검색하고 관련성 점수를 확인하세요
# 힌트: for 루프로 test_queries 순회, compression_retriever.invoke(test_query)
# 예상 결과: 각 쿼리별 상위 3개 문서와 Jina 관련성 점수 출력
# ============================================================

if compression_retriever is None:
    print("⚠️ Jina Reranker가 없어 추가 테스트를 건너뜁니다.")
else:
    # 다양한 쿼리로 Jina Reranker 성능 검증
    test_queries = [
        "비즈니스를 디지털 기술로 혁신하는 과정은?",
        "벡터로 변환된 데이터를 효율적으로 보관하는 시스템은?",
        "소프트웨어 코드가 공개되어 자유롭게 사용할 수 있는 것은?"
    ]

    print("\n" + "=" * 100)
    print("🧪 Jina Reranker 다양한 쿼리 테스트")
    print("=" * 100)

    for idx, test_query in enumerate(test_queries, 1):
        print(f"\n{'─' * 100}")
        print(f"[테스트 {idx}] 쿼리: {test_query}")
        print(f"{'─' * 100}")
        
        # Jina Reranker 적용
        reranked_docs = compression_retriever.invoke(test_query)
        
        print(f"\n✅ Jina가 선택한 상위 {len(reranked_docs)}개 문서:")
        
        for i, doc in enumerate(reranked_docs, 1):
            preview = doc.page_content.replace('\n', ' ')[:80]
            score = doc.metadata.get('relevance_score', 0)
            print(f"\n  [{i}] 점수: {score:.4f}")
            print(f"      내용: {preview}...")


🧪 Jina Reranker 다양한 쿼리 테스트

────────────────────────────────────────────────────────────────────────────────────────────────────
[테스트 1] 쿼리: 비즈니스를 디지털 기술로 혁신하는 과정은?
────────────────────────────────────────────────────────────────────────────────────────────────────

✅ Jina가 선택한 상위 3개 문서:

  [1] 점수: 0.4683
      내용: HuggingFace  정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하는 라이브러리입니다. 이는 연구...

  [2] 점수: 0.1082
      내용: Open Source  정의: 오픈 소스는 소스 코드가 공개되어 누구나 자유롭게 사용, 수정, 배포할 수 있는 소프트웨어를 의미합니다. 이는 협...

  [3] 점수: 0.0827
      내용: 멀티모달 (Multimodal)  정의: 멀티모달은 여러 종류의 데이터 모드(예: 텍스트, 이미지, 소리 등)를 결합하여 처리하는 기술입니다. ...

────────────────────────────────────────────────────────────────────────────────────────────────────
[테스트 2] 쿼리: 벡터로 변환된 데이터를 효율적으로 보관하는 시스템은?
────────────────────────────────────────────────────────────────────────────────────────────────────

✅ Jina가 선택한 상위 3개 문서:

  [1] 점수: 0.5450
      내용: Token  정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다. 예시

## 7. 핵심 정리

### Reranker 비교

| 특징 | Jina Reranker | Cohere Rerank | Cross-Encoder (로컬) |
|------|:---:|:---:|:---:|
| 최대 토큰 | **8K** | 4K | 512 |
| 속도 | 빠름 | 빠름 | 보통 |
| 정확도 | 높음 | 최고 수준 | 높음 |
| 비용 | $ | $$ | 무료 |
| 추천 용도 | **긴 문서** | 프로덕션 | 개발/테스트 |

### 기본 사용법

```python
from langchain_community.document_compressors import JinaRerank

compressor = JinaRerank(
    model="jina-reranker-v2-base-multilingual",
    top_n=3,
)
```

### 주요 파라미터

| 파라미터 | 설명 | 기본값 |
|---------|------|--------|
| `model` | Reranker 모델명 | `jina-reranker-v2-base-multilingual` |
| `top_n` | 최종 반환 문서 수 | 3 |
| `jina_api_key` | Jina API 키 | 환경변수 `JINA_API_KEY` |

## 마무리 정리 — Reranker 시리즈 완성

이 시리즈(01~03)에서 배운 세 가지 Reranker를 정리해요:

| Reranker | 환경 | 토큰 한도 | 비용 | 추천 상황 |
|---------|------|:---:|:---:|----------|
| **Cross-Encoder** | 로컬 | 512 | 무료 | 개발·테스트, 데이터 보안 |
| **Cohere** | 클라우드 | 4K | $$ | 한국어 프로덕션 |
| **Jina** | 클라우드 | **8K** | $ | 긴 문서, 비용 효율 |

### 이제 무엇을 배우나요?

다음 6-7 RAG Process 섹션에서는 지금까지 배운 Retriever, Reranker를 모두 조합해 완전한 End-to-End RAG 시스템을 구축해요. PDF, 웹, 대화형, RAPTOR 등 다양한 시나리오를 직접 구현해봅니다.